# Препроцессинг .rpt-отчётов Abaqus → 4-компонентный тензор

Сборка датасета остаточных напряжений и накопленных деформаций по итогам осесимметричного МКЭ-моделирования волочения проволоки. Из каждого `.rpt`-файла извлекается профиль вдоль радиуса, отфильтрованный по 25-75% высоты сечения (зона установившегося течения), и усредняется до 20 узлов. Параметры процесса считываются из имени файла.

На выходе получаем четыре массива:
- `X_stress`, `X_strain`: shape `(4, N, 5)` - входные параметры `[red, cal, ha, fric, vel]` для каждой из 4 компонент;
- `y_stress`, `y_strain`: shape `(4, N, 20)` - радиальный профиль значений компонент.

Порядок компонент в обоих случаях: `[rr, θθ, zz, rz]`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile

zip_path = '/content/drive/MyDrive/Raw_Data.zip'
extract_path = '/content/drive/MyDrive/Raw_Data'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

## Модуль `rpt_utils.py`

Парсер `.rpt`-формата, агрегация профиля до 20 узлов, извлечение параметров из имени файла, сборка итоговых массивов. Формат `.rpt` фиксирован: 15 числовых колонок - `Node`, `COOR1`, `COOR2`, `LE11`, `LE22`, `LE33`, `LE12`, `Mises`, `S.Max`, `S.Mid`, `S.Min`, `S11`, `S22`, `S33`, `S12`. Принятое осесимметричное соответствие: `1=r`, `2=z`, `3=θ`, поэтому `S11→σ_rr`, `S22→σ_zz`, `S33→σ_θθ`, `S12→τ_rz` (аналогично для `LE`, причём `LE12` - инженерный сдвиг, тензорная компонента `ε_rz = LE12/2`).

In [ ]:
%%writefile /content/rpt_utils.py
import os
import pickle
from collections import Counter

import numpy as np


def saver(obj, name, path_import):
    out = os.path.join(path_import, name + '.pkl')
    with open(out, 'wb') as f:
        pickle.dump(obj, f)
    print('saved in ' + out)


def read_rpt_np(path, expected_cols=15):
    rows = []
    with open(path, 'r', errors='ignore') as fh:
        for ln in fh:
            toks = ln.strip().split()
            if not toks:
                continue
            try:
                vals = [float(t) for t in toks]
            except ValueError:
                continue
            rows.append(vals)
    if not rows:
        raise ValueError(f'{path}: пусто')
    ncols, _ = Counter(len(r) for r in rows).most_common(1)[0]
    rows = [r for r in rows if len(r) == ncols]
    arr = np.asarray(rows, dtype='double')
    if arr.shape[1] != expected_cols:
        raise ValueError(f'{path}: ожидал {expected_cols} колонок, получил {arr.shape[1]}')
    return arr


def preprocessing_res_np(file):
    x_1 = file[:, 1] * 1e3
    y_1 = file[:, 2] * 1e3

    e_rr = file[:, 3]
    e_zz = file[:, 4]
    e_tt = file[:, 5]
    e_rz = 0.5 * file[:, 6]

    s_rr = file[:, 11] / 1e6
    s_zz = file[:, 12] / 1e6
    s_tt = file[:, 13] / 1e6
    s_rz = file[:, 14] / 1e6

    lo = 0.25 * (np.max(y_1) - np.min(y_1))
    hi = 0.75 * (np.max(y_1) - np.min(y_1))
    mask = (y_1 >= lo) & (y_1 <= hi)

    x_sel = x_1[mask]
    y_sel = y_1[mask]

    Stress = np.stack([s_rr[mask], s_tt[mask], s_zz[mask], s_rz[mask]], axis=-1)
    Strain = np.stack([e_rr[mask], e_tt[mask], e_zz[mask], e_rz[mask]], axis=-1)

    order = np.argsort(x_sel)
    return x_sel[order], y_sel[order], Stress[order], Strain[order]


def average_val_np(arr, nodes=20):
    b = np.zeros(nodes)
    chunks = np.array_split(arr, nodes)
    for i in range(nodes):
        b[i] = np.median(chunks[i])
    return b


def do_rpt_list(path_import):
    names = []
    for fn in os.listdir(path_import):
        if fn.endswith('.rpt'):
            names.append(fn[:-4])
    names.sort()
    return names


def do_preprocessing_rpt(names, path_import):
    out = []
    for nm in names:
        arr = read_rpt_np(os.path.join(path_import, nm + '.rpt'))
        out.append(preprocessing_res_np(arr))
    print(f'  {len(out)} files processed')
    return out


def _split(j):
    return j.split('_')


def r(j):
    s = _split(j)
    for i, t in enumerate(s):
        if t in ('red', 'rd'):
            return float(s[i + 1]) / 10000


def c(j):
    s = _split(j)
    for i, t in enumerate(s):
        if t == 'cal':
            return float(s[i + 1]) / 100


def f(j):
    s = _split(j)
    for i, t in enumerate(s):
        if t in ('fric', 'f'):
            token = s[i + 1].split('.')[0]
            return float(token[1:]) / 1000


def v(j):
    s = _split(j)
    for i, t in enumerate(s):
        if t in ('vel', 'v'):
            return int(s[i + 1])


def h(j):
    s = _split(j)
    for i, t in enumerate(s):
        if t == '2a':
            return int(s[i + 1]) / 2


def get_param(cur_job_name, job_list, all_arrays, char_1=2, char_2=0):
    idx = job_list.index(cur_job_name)
    char = average_val_np(all_arrays[idx][char_1][:, char_2], 20)
    return [r(cur_job_name), c(cur_job_name), h(cur_job_name),
            v(cur_job_name), f(cur_job_name), char]


def data_preparer(train_list, train_arrays):
    X_stress, X_strain = [], []
    y_stress, y_strain = [], []
    for char in range(2):
        for comp in range(4):
            X = np.zeros((len(train_list), 5))
            y = np.zeros((len(train_list), 20))
            for i, jn in enumerate(train_list):
                red, cal, ha, vel, fric, val = get_param(
                    jn, train_list, train_arrays, char_1=2 + char, char_2=comp)
                X[i] = [red, cal, ha, fric, vel]
                y[i] = val
            (X_stress if char == 0 else X_strain).append(X)
            (y_stress if char == 0 else y_strain).append(y)
    return (np.array(X_stress), np.array(X_strain),
            np.array(y_stress), np.array(y_strain))

## Структура исходных данных

Исходные `.rpt`-отчёты лежат в `Raw_Data` группами по скоростям волочения: `Vel_5`, `Vel_10`, `Vel_20`, `Vel_40`, `Vel_250`.

In [ ]:
import os

ROOT = '/content/drive/MyDrive/Raw_Data'
for dirpath, dirnames, filenames in os.walk(ROOT):
    depth = dirpath.replace(ROOT, '').count(os.sep)
    if depth > 2:
        continue
    print(f"{'  ' * depth}{os.path.basename(dirpath) or ROOT}/  [{len(dirnames)} dirs, {len(filenames)} files]")

In [ ]:
import sys
sys.path.insert(0, '/content')

import importlib
import rpt_utils
importlib.reload(rpt_utils)
from rpt_utils import (preprocessing_res_np, data_preparer, saver,
                       do_rpt_list, do_preprocessing_rpt)

## Сбор отчётов по всем папкам скоростей

In [ ]:
BASE = '/content/drive/MyDrive/Raw_Data'
VEL_DIRS = ['Vel_5', 'Vel_10', 'Vel_20', 'Vel_40', 'Vel_250']

all_files, all_arrays = [], []
per_vel_counts = {}

for vd in VEL_DIRS:
    p = os.path.join(BASE, vd)
    if not os.path.isdir(p):
        print(f'пропущено: {p}')
        continue
    print(f'→ {vd}')
    names = do_rpt_list(p)
    per_vel_counts[vd] = len(names)
    arrays = do_preprocessing_rpt(names, p)
    all_files.extend(names)
    all_arrays.extend(arrays)

print('\nфайлов по скоростям:', per_vel_counts)
print('всего файлов:', len(all_files))

## Сборка датасета и сохранение

In [ ]:
Xs, Xe, ys, ye = data_preparer(all_files, all_arrays)
print('X_stress:', Xs.shape)
print('X_strain:', Xe.shape)
print('y_stress:', ys.shape)
print('y_strain:', ye.shape)

OUT = '/content/Raw_Data_pkl'
os.makedirs(OUT, exist_ok=True)
POSTFIX = 'new_le_ss_full'

saver(Xs, f'X_stress_components_{POSTFIX}', path_import=OUT)
saver(Xe, f'X_strain_components_{POSTFIX}', path_import=OUT)
saver(ys, f'y_stress_components_{POSTFIX}', path_import=OUT)
saver(ye, f'y_strain_components_{POSTFIX}', path_import=OUT)

## Базовая проверка

In [ ]:
from collections import Counter
import numpy as np

X = Xs[0]
cols = ['red', 'cal', 'ha', 'fric', 'vel']
for i, col in enumerate(cols):
    u = np.unique(X[:, i])
    print(f'{col:5s}: n={len(u)}  {u}')

print('\nпо скоростям:')
for vel, n in sorted(Counter(X[:, 4].tolist()).items()):
    print(f'  vel={int(vel):3d}: {n} строк')

n_red = len(np.unique(X[:, 0]))
n_cal = len(np.unique(X[:, 1]))
n_ha = len(np.unique(X[:, 2]))
n_fric = len(np.unique(X[:, 3]))
grid = n_red * n_cal * n_ha * n_fric
print(f'\nполная решётка red×cal×ha×fric = {n_red}×{n_cal}×{n_ha}×{n_fric} = {grid}')
print(f'дубликатов X: {X.shape[0] - np.unique(X, axis=0).shape[0]}')
print(f'NaN в y_stress: {np.isnan(ys).any()},  в y_strain: {np.isnan(ye).any()}')

In [ ]:
for k, name in enumerate(['rr', 'θθ', 'zz', 'rz']):
    print(f'σ_{name}: min={ys[k].min():+.1f}  mean={ys[k].mean():+.1f}  max={ys[k].max():+.1f}')
print()
for k, name in enumerate(['rr', 'θθ', 'zz', 'rz']):
    print(f'ε_{name}: min={ye[k].min():+.3f}  mean={ye[k].mean():+.3f}  max={ye[k].max():+.3f}')

## Перенос pkl на Drive

In [ ]:
DRIVE_OUT = '/content/drive/MyDrive/PINN_data_pkl'
os.makedirs(DRIVE_OUT, exist_ok=True)
!cp /content/Raw_Data_pkl/*.pkl "$DRIVE_OUT/"
!ls -lh "$DRIVE_OUT/"